# D1 local ablation (qwen3.5:4b via Ollama) on Colab

Runs `scripts/local_ablation.py` — the V1 local tier arm `D1_local_single` — against Ollama's `qwen3.5:4b` inside a Colab runtime instead of your laptop.

**Before running:** Runtime → Change runtime type → pick a GPU (T4 is fine, free tier). CPU-only will also work, just slower.

Only the `D1` arm is implemented locally today (single local agent). The hosted `A_deterministic` / `C_crew_llm` arms are a separate, not-yet-built piece of work — out of scope here.

Order: install Ollama → pull the model → get the repo onto the runtime → install Python deps → fetch external corpora → build the dev manifest → `freeze` → `run` → `summarise`.

## 1. Install and start Ollama

In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
import subprocess, time, urllib.request, urllib.error
import sys

print("Starting ollama daemon...")
ollama_proc = subprocess.Popen(["ollama", "serve"], stdout=subprocess.PIPE, stderr=subprocess.PIPE)
time.sleep(5)

for attempt in range(120):
    try:
        urllib.request.urlopen("http://127.0.0.1:11434/api/version", timeout=2)
        print(f"✓ ollama daemon is up (attempt {attempt + 1})")
        break
    except Exception as e:
        if attempt % 10 == 0:
            print(f"  waiting... (attempt {attempt + 1}, error: {type(e).__name__})")
        time.sleep(1)
else:
    print("✗ ollama daemon did not start")
    stderr = ollama_proc.stderr.read().decode() if ollama_proc.stderr else "no stderr"
    print("Stderr:", stderr)
    sys.exit(1)

In [ ]:
!ollama pull qwen3.5:4b

## 2. Get the repo onto the runtime

The repo is a **private** GitHub repo (`<private-owner>/agentic-threat-hunter`), so a plain `git clone` will 404. Two options — use whichever is easier:

**Option A — clone with a token.** Create a fine-grained GitHub PAT with read access to this one repo, add it as a Colab secret named `GH_TOKEN` (key icon in the left sidebar), then run the cell below.

**Option B — upload a zip.** Zip your local working copy (skip `.git`, `data/external/`, `reports/local/`) and upload/mount it, then skip the clone cell and just `%cd` into the extracted folder.

In [ ]:
# Option A: clone with a token stored as a Colab secret
from google.colab import userdata

GH_TOKEN = userdata.get("GH_TOKEN")
REPO = "<private-owner>/agentic-threat-hunter"
BRANCH = "m14-real-data-validation"

!apt-get -qq install -y git-lfs && git lfs install
!git clone -b {BRANCH} https://{GH_TOKEN}@github.com/{REPO}.git /content/agentic-threat-hunter
!cd /content/agentic-threat-hunter && git lfs pull

In [ ]:
%cd /content/agentic-threat-hunter

If you're continuing a run across sessions, point this at Drive instead so `reports/local/dev/rows/` (already-completed rows are skipped, not re-run) and `data/external/` (large corpora, slow to refetch) survive a runtime reset:
```python
from google.colab import drive
drive.mount('/content/drive')
# then symlink or copy reports/local and data/external to/from /content/drive/MyDrive/...
```

## 3. Python deps

In [ ]:
!pip install -q -r requirements.txt
!pip install -q -e .

## 4. Fetch external corpora + build the dev manifest

`data/external/` is gitignored (only the manifest index is committed) so it has to be fetched fresh on a new runtime — no credentials required, these are public datasets pulled by URL.

In [ ]:
!python scripts/fetch_external.py

In [ ]:
!python scripts/local_manifest.py build

## 5. freeze / run / summarise

`freeze` pins the model digest + daemon version against the manifest hash; `run` refuses if the live daemon has drifted from that freeze. `run` is resumable — rerunning the cell after a Colab disconnect skips rows already written to `reports/local/dev/rows/D1_qwen3.5-4b/`.

In [ ]:
!python scripts/local_ablation.py freeze --model qwen3.5:4b --arm D1

In [ ]:
!python scripts/local_ablation.py run --model qwen3.5:4b --arm D1 --repeat 1 --seed 0

In [ ]:
!python scripts/local_ablation.py summarise --model qwen3.5:4b --arm D1 --repeat 1

In [ ]:
print(open("reports/local/dev/SUMMARY_D1_qwen3.5-4b_rep1.md").read())

## 6. (optional) pull results back down

Zip `reports/local/dev/` and download it, or copy it to a mounted Drive, so the rows survive the Colab runtime being recycled.

In [ ]:
!zip -r -q /content/dev_results.zip reports/local/dev
from google.colab import files
files.download("/content/dev_results.zip")